# PWV01seasons : Seasons
**Goal :** Show seasons for PWV

- author Sylvie Dagoret-Campagne
- creation date 2026-03-18 : version run2026_v02b_cr_run2026_v02d_cr
- adapted to the 2026 merged files (runs 2023-2026 from Corentin Ravoux) on 2026-11-13
- affiliation : IJCLab
- Kernel @usdf **w_2026_02**
- laptop : conda_py313 (python 3.13), package `mysitcom` installed with `pip install -e .` at the top level of `mySIT-COM2025`

**Goal** : Show PWV and diffPWV (`ram` - `rum`) versus time and histograms, and compare without and with quality cuts.
- `ram` : PWV fitted on the spectrogram (2D image of the CCD)
- `rum` : PWV fitted on the 1D spectrum (projection on the dispersion axis)

Main changes with respect to the 2026-03-19 version :
- input = merged files of `../2026-11-13-SpectractorExtraction-FromButler` (no more butler collection bookkeeping);
- `diff_PWV_err` is computed in quadrature (sum of the squares of the errors, the 2026-03 version subtracted them);
- the filters for which no cut is defined in the json file (`FELH0600`, `SDSSr`) are removed after the cuts (flag `FLAG_KEEP_ONLY_FILTERS_WITH_CUTS`), otherwise they would pass all the cuts;
- the plots announced in the goal are done (PWV, diffPWV, histograms, before and after cuts), the selected data are saved for the next notebooks.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import warnings
from collections import OrderedDict
from platform import python_version

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from astropy.time import Time

warnings.simplefilter('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

plt.rcParams.update({
    'figure.figsize': (16, 8),
    'axes.labelsize': 'xx-large',
    'axes.titlesize': 'xx-large',
    'xtick.labelsize': 'xx-large',
    'ytick.labelsize': 'xx-large',
})
print(f'python {python_version()}, numpy {np.__version__}, pandas {pd.__version__}')

In [ ]:
# must install the mysitcom package by doing at top level "pip install --user -e ."
from mysitcom.auxtel.pwv import (
    strip_datetime,
    plot_atmparam_vs_time, plot_atmparam_diff_vs_time,
    plot_atmparam_hist_per_filter, plot_atmparam_diff_hist_per_filter,
    normalize_column_data_bytarget_byfilter,
    GetNightMidnightsDict, GetNightBoundariesDict,
)
from mysitcom.auxtel.qualitycuts import ParameterCutSelection, ParameterCutTools

## Configuration

In [ ]:
from PWV00_parameters import (
    version_run, atmfilename, tag,
    FLAG_PWVFILTERS, PWV_FILTER_LIST,
    PWV_COL_RAM, PWV_ERR_COL_RAM, PWV_COL_RUM, PWV_ERR_COL_RUM,
    DumpConfig,
)
DumpConfig()

In [ ]:
# where are stored the figures and the data
NB_TAG = 'PWV01seas'
prefix = 'pwv01seas'
pathfigs = f'figs_{NB_TAG}'
pathdata = f'data_{NB_TAG}'
for path in (pathfigs, pathdata):
    os.makedirs(path, exist_ok=True)


def save_fig(fig, name, formats=('png', 'pdf')):
    '''Save a figure in the figure directory, with the notebook prefix and the version of the run.'''
    for ext in formats:
        fig.savefig(f'{pathfigs}/{prefix}_{version_run}_{name}.{ext}', bbox_inches='tight')

In [ ]:
PWVMIN = 0.
PWVMAX = 16.

# the collimator was installed on 2023-09-30
FLAG_WITHCOLLIMATOR = False
datetime_WITHCOLLIMATOR = pd.to_datetime('2023-09-30 00:00:00.0+0000')

# type of quality cuts (json files in data_PWV01seas)
FLAG_LOOSE_CUTS = True
FLAG_TIGHT_CUTS = False

# the json files define cuts for empty, OG550_65mm_1 and BG40_65mm_1 only :
# True = remove from the selected data the filters without cuts (FELH0600, SDSSr)
FLAG_KEEP_ONLY_FILTERS_WITH_CUTS = True

# night boundaries and midnights as vertical lines in the plots versus time
FLAG_NIGHT_VERTICALLINES = True

FILTER_COLORS = {'empty': 'blue',
                 'BG40_65mm_1': 'green',
                 'BG40': 'green',
                 'OG550_65mm_1': 'red',
                 'OG550': 'red',
                 'FELH0600': 'purple',
                 }

## Initialisation
### Read the file
`atmfilename` is defined in `PWV00_parameters.py`.

In [ ]:
print(f'INPUT FILENAME : {atmfilename}')
if atmfilename.endswith('parquet') or atmfilename.endswith('parquet.gz'):
    df_spec = pd.read_parquet(atmfilename)
else:
    df_spec = pd.DataFrame(np.load(atmfilename, allow_pickle=True))

# the two methods have the same chi2 (CHI2_FIT), the columns are expected by the quality cuts
for col in ('chi2_ram', 'chi2_rum'):
    if col not in df_spec.columns:
        df_spec[col] = df_spec['CHI2_FIT']

print(f'{len(df_spec)} spectra, {len(df_spec.columns)} columns')

In [ ]:
# time, sorted in chronological order
df_spec['Time'] = pd.to_datetime(df_spec['DATE-OBS'], utc=True)
df_spec = df_spec.sort_values(by='Time', ascending=True).reset_index(drop=True)
df_spec['seq_num'] = df_spec['id'] % 100_000

# night of observation : compare the id with the local time in Chile (night = local date of time - 12 h)
df_spec['nightObs_id'] = df_spec['id'] // 100_000
df_spec['Time_local'] = df_spec['Time'].dt.tz_convert('America/Santiago')
df_spec['nightObs_corr'] = (df_spec['Time_local'] - pd.Timedelta(hours=12)).dt.strftime('%Y%m%d').astype(int)
n_mismatch = (df_spec['nightObs_id'] != df_spec['nightObs_corr']).sum()
print(f'night from the id different from the night computed from the local time : {n_mismatch} / {len(df_spec)}')

FLAG_CORRECT_NIGHTOBS_VARIABLES = True
df_spec['nightObs'] = df_spec['nightObs_corr'] if FLAG_CORRECT_NIGHTOBS_VARIABLES else df_spec['nightObs_id']
span = df_spec.groupby('nightObs')['Time'].max() - df_spec.groupby('nightObs')['Time'].min()
print(f'longest night : {span.max()}')

## Select the filters

In [ ]:
print('filters in the file :', df_spec['FILTER'].unique())
unknown = set(df_spec['FILTER'].unique()) - set(FILTER_COLORS)
assert not unknown, f'add these filters to FILTER_COLORS : {unknown}'

if FLAG_PWVFILTERS:
    df_spec = df_spec[df_spec['FILTER'].isin(PWV_FILTER_LIST)]
if FLAG_WITHCOLLIMATOR:
    df_spec = df_spec[df_spec['Time'] > datetime_WITHCOLLIMATOR]
df_spec = df_spec.reset_index(drop=True)
print('filters kept :', df_spec['FILTER'].unique(), f', {len(df_spec)} spectra')

### Check filters

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 4))
strip_datetime(df=df_spec, x='Time', y='FILTER', hue='FILTER', ax=ax, size=9)
ax.set_title(tag)
save_fig(fig, 'filters_vs_time')
plt.show()

## Palette with more clear target separation colors

In [ ]:
target_counts = df_spec['TARGET'].value_counts().sort_values(ascending=False)
targets = target_counts.index.tolist()
counts = target_counts.values

palette = (sns.color_palette('tab20b', 20) + sns.color_palette('tab20c', 10))[:len(targets)]
assert len(palette) == len(targets), 'more than 30 targets : choose a larger palette'
target_color_map = OrderedDict(zip(targets, palette))

cmap = mpl.colors.ListedColormap(palette)
norm = mpl.colors.BoundaryNorm(boundaries=range(len(targets) + 1), ncolors=len(targets))

In [ ]:
fig, ax = plt.subplots(figsize=(max(0.8 * len(targets), 6), 1.8), layout='constrained')
cb = mpl.colorbar.ColorbarBase(ax, cmap=cmap, norm=norm, orientation='horizontal')
cb.set_ticks([i + 0.5 for i in range(len(targets))])
cb.set_ticklabels(targets)
cb.ax.tick_params(labelrotation=90, labelsize=10, length=6, width=1.5)
cb.set_label('TARGET', labelpad=10)
plt.suptitle(tag)
save_fig(fig, 'palette_targetnames')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 0.35 * len(targets) + 1), layout='constrained')
sns.barplot(x=counts, y=targets, hue=targets, palette=palette, legend=False, ax=ax)
ax.set_xlabel('Number of Obs')
ax.set_ylabel('TARGET')
ax.set_title('TARGET observed')
ax.tick_params(axis='y', labelsize=10)
ax.grid()
plt.suptitle(tag)
save_fig(fig, 'barplot_targets')
plt.show()

## Processing of some quantities
### Difference of PWV between the spectrogram (`ram`) and the spectrum (`rum`)

In [ ]:
err_quad = np.sqrt(df_spec[PWV_ERR_COL_RAM]**2 + df_spec[PWV_ERR_COL_RUM]**2)

df_spec['diff_PWV'] = df_spec[PWV_COL_RAM] - df_spec[PWV_COL_RUM]
# the errors of the two methods are added in quadrature
df_spec['diff_PWV_err'] = err_quad
df_spec['diff_PWV_norm'] = np.where(np.isfinite(err_quad) & (err_quad > 0), df_spec['diff_PWV'] / err_quad, np.nan)

### Normalized chi2

In [ ]:
df_spec, _ = normalize_column_data_bytarget_byfilter(df_spec, target_col='TARGET', filter_col='FILTER', feature_col='CHI2_FIT', ext='norm')
df_spec, _ = normalize_column_data_bytarget_byfilter(df_spec, target_col='TARGET', filter_col='FILTER', feature_col='chi2_ram', ext='norm')
df_spec, _ = normalize_column_data_bytarget_byfilter(df_spec, target_col='TARGET', filter_col='FILTER', feature_col='chi2_rum', ext='norm')

## Apply quality cuts
### Parameters cuts load

In [ ]:
if FLAG_LOOSE_CUTS:
    filename_cuts_final = f'{pathdata}/cuts_loose_finaldecision.json'
    cutstype_name = 'loose-cuts'
elif FLAG_TIGHT_CUTS:
    filename_cuts_final = f'{pathdata}/cuts_tight_finaldecision.json'
    cutstype_name = 'tight-cuts'
else:
    filename_cuts_final = f'{pathdata}/cuts_finaldecision.json'
    cutstype_name = 'standard-cuts'

cuts = ParameterCutTools.load_cuts_json(filename_cuts_final)
list_of_params = list(cuts.keys())
filters_with_cuts = sorted({filt for per_filter in cuts.values() for filt in per_filter})
print(f'{cutstype_name} : {len(list_of_params)} parameters, filters with cuts : {filters_with_cuts}')
ParameterCutTools.cuts_to_dataframe(cuts).head(12)

### Selection

In [ ]:
selector = ParameterCutSelection(df_spec, params=list_of_params, id_col='id')
flags = selector.apply_cuts(cuts)
df_stats = selector.selection_statistics(cuts)

df_selected = df_spec.merge(flags, on='id')
df_keep = df_selected[df_selected['pass_all_cuts']].copy()

if FLAG_KEEP_ONLY_FILTERS_WITH_CUTS:
    removed = df_keep[~df_keep['FILTER'].isin(filters_with_cuts)]
    if len(removed):
        print(f'filters without cuts removed : {removed["FILTER"].value_counts().to_dict()}')
    df_keep = df_keep[df_keep['FILTER'].isin(filters_with_cuts)].copy()

N_init = len(df_spec)
N_keep = len(df_keep)
print(f'Selection results ({cutstype_name}) {N_keep} passed / {N_init}')

In [ ]:
# selection efficiency per filter
eff = pd.DataFrame({'N total': df_spec.groupby('FILTER').size(), 'N selected': df_keep.groupby('FILTER').size()}).fillna(0).astype(int)
eff['efficiency [%]'] = (100. * eff['N selected'] / eff['N total']).round(1)
eff

In [ ]:
# number of spectra per month and per filter, before and after the cuts
fig, axs = plt.subplots(2, 1, figsize=(18, 9), sharex=True, layout='constrained')
for ax, d, label in zip(axs, (df_spec, df_keep), ('without cuts', f'with {cutstype_name}')):
    month = d['Time'].dt.strftime('%Y-%m')
    cnt = d.groupby([month, 'FILTER']).size().unstack(fill_value=0)
    cnt.plot(kind='bar', stacked=True, ax=ax, color=[FILTER_COLORS[c] for c in cnt.columns], width=0.9)
    ax.set_title(f'Number of spectra per month, {label}')
    ax.set_ylabel('N')
    ax.grid(axis='y', alpha=0.3)
axs[1].set_xlabel('month')
plt.suptitle(tag)
save_fig(fig, 'counts_per_month')
plt.show()

## Night boundaries and midnights
Takes about 0.3 s per night.

In [ ]:
dn = GetNightBoundariesDict(df_spec)
dnidnights = GetNightMidnightsDict(df_spec)

## PWV versus time, without and with quality cuts

In [ ]:
def add_night_lines(ax):
    '''Vertical lines at the astronomical midnight of each night.'''
    if FLAG_NIGHT_VERTICALLINES:
        for midnight in dnidnights.values():
            ax.axvline(pd.Timestamp(midnight).tz_localize(None) if pd.Timestamp(midnight).tzinfo else midnight,
                       color='purple', ls=':', lw=0.4, alpha=0.3)


def plot_before_after(param_col, err_col, name, ylim=(PWVMIN, PWVMAX)):
    '''A parameter versus time : top without cuts, bottom with cuts.'''
    fig, axs = plt.subplots(2, 1, figsize=(18, 10), sharex=True, layout='constrained')
    for ax, d, label in zip(axs, (df_spec, df_keep), ('without cuts', f'with {cutstype_name}')):
        plot_atmparam_vs_time(d, 'Time', 'FILTER', param_col, param_err_col=err_col,
                              param_min_fig=ylim[0], param_max_fig=ylim[1],
                              title_param=f'{param_col}, {label}', axs=ax)
        add_night_lines(ax)
    plt.suptitle(tag)
    save_fig(fig, name)
    plt.show()


plot_before_after(PWV_COL_RAM, PWV_ERR_COL_RAM, 'pwv_ram_vs_time')

In [ ]:
plot_before_after(PWV_COL_RUM, PWV_ERR_COL_RUM, 'pwv_rum_vs_time')

## Difference of PWV (`ram` - `rum`) versus time, without and with quality cuts

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(18, 10), sharex=True, layout='constrained')
for ax, d, label in zip(axs, (df_spec, df_keep), ('without cuts', f'with {cutstype_name}')):
    plot_atmparam_diff_vs_time(d, 'Time', 'FILTER', PWV_COL_RAM, PWV_COL_RUM,
                               param1_err_col=PWV_ERR_COL_RAM, param2_err_col=PWV_ERR_COL_RUM,
                               param_min_fig=-4., param_max_fig=4.,
                               title_param=f'PWV ram - rum, {label}', axs=ax)
    add_night_lines(ax)
plt.suptitle(tag)
save_fig(fig, 'diffpwv_vs_time')
plt.show()

## Histograms per filter, without and with quality cuts

In [ ]:
# one figure per panel : the library puts the statistics of each filter on the right of the axes
hist_params = [
    ('pwv_ram', PWV_COL_RAM, None, (PWVMIN, PWVMAX)),
    ('pwv_rum', PWV_COL_RUM, None, (PWVMIN, PWVMAX)),
    ('diffpwv', None, None, (-3., 3.)),
]
for name, col, _, rng in hist_params:
    for d, label, suffix in ((df_spec, 'without cuts', 'nocuts'), (df_keep, f'with {cutstype_name}', 'cuts')):
        fig, ax = plt.subplots(1, 1, figsize=(11, 6))
        if col is not None:
            plot_atmparam_hist_per_filter(d, 'FILTER', col, rng, param_min_fig=rng[0], param_max_fig=rng[1],
                                          title_param=f'{col}, {label}', suptitle=tag, axs=ax)
        else:
            plot_atmparam_diff_hist_per_filter(d, 'FILTER', PWV_COL_RAM, PWV_COL_RUM, rng, param_min_fig=rng[0], param_max_fig=rng[1],
                                               title_param=f'PWV ram - rum, {label}', suptitle=tag, axs=ax)
        save_fig(fig, f'hist_{name}_{suffix}')
        plt.show()

## Save the selected data for the next notebooks

In [ ]:
cols_drop = ['Time_local', 'nightObs_id', 'nightObs_corr']
df_out = df_keep.drop(columns=[c for c in cols_drop if c in df_keep.columns]).reset_index(drop=True)

filename_out = f'{pathdata}/auxtel_atmosphere_{version_run}_{cutstype_name}_selected'
try:
    df_out.to_parquet(filename_out + '.parquet')
    print('saved', filename_out + '.parquet')
except ImportError:
    df_out.to_pickle(filename_out + '.pkl')
    print('pyarrow not available, saved', filename_out + '.pkl')
print(f'{len(df_out)} selected spectra, {df_out["nightObs"].nunique()} nights')